# GPU benchmark: first-order inner solvers in aulax

Run on Google Colab with **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
The notebook installs aulax from the `first-order-inner-improvements` branch, checks the GPU, smoke-tests the first-order inner solvers, runs the benchmark suites on the GPU, and downloads the result files as a zip at the end.


In [ ]:
!pip install -q "jax[cuda12]"
!git clone --depth 1 --branch first-order-inner-improvements https://github.com/nardi/aulax.git /content/aulax
!pip install -q -e /content/aulax optax scipy nlopt

In [ ]:
import jax
import aulax
import optax
import optimistix

print("jax", jax.__version__, jax.devices())
print("aulax", aulax.__file__)
print("optax", optax.__version__, "optimistix", optimistix.__version__)

## Smoke test

Each candidate as the inner minimiser of the augmented Lagrangian on the chain problem, under `jax.jit`, plus a `jax.grad` and a `jax.vmap` check.


In [ ]:
import jax.numpy as jnp
import optimistix as optx

import aulax as alx

jax.config.update("jax_enable_x64", True)


def chain(y, args, state):
    return alx.Evaluation(
        objective=jnp.sum((y - args) ** 2) + 0.1 * jnp.sum(jnp.diff(y) ** 2),
        equality=y[1:] - 0.5 * y[:-1] - 0.1 * jnp.sin(y[1:]),
        inequality=jnp.concatenate([-y - 1.0, y - 2.0]),
    )


target = jnp.linspace(0.0, 1.0, 20)

inners = {
    "nonlinear_cg": optx.NonlinearCG(rtol=1e-8, atol=1e-8),
    "dog": optx.OptaxMinimiser(optax.contrib.dog(), rtol=1e-8, atol=1e-8),
    "prodigy": optx.OptaxMinimiser(optax.contrib.prodigy(), rtol=1e-8, atol=1e-8),
    "dadapt_adamw": optx.OptaxMinimiser(
        optax.contrib.dadapt_adamw(), rtol=1e-8, atol=1e-8
    ),
    "lion_1e-3": optx.OptaxMinimiser(
        optax.lion(learning_rate=1e-3), rtol=1e-8, atol=1e-8
    ),
    "lion_1e-4": optx.OptaxMinimiser(
        optax.lion(learning_rate=1e-4), rtol=1e-8, atol=1e-8
    ),
}

for name, inner in inners.items():
    solver = alx.AugmentedLagrangian(inner=inner)
    solve = jax.jit(
        lambda: alx.minimise(
            chain, solver, jnp.zeros(20), target, max_steps=4096, throw=False
        )
    )
    solution = jax.block_until_ready(solve())
    print(
        name,
        "result=",
        solution.result.name,
        "infeasibility=",
        float(solution.stats.infeasibility),
    )

solver = alx.AugmentedLagrangian(inner=inners["dog"])


def objective_sum(t):
    solution = alx.minimise(
        chain, solver, jnp.zeros(20), t, max_steps=4096, throw=False
    )
    return jnp.sum(solution.value)


print("grad ok:", float(jnp.sum(jax.grad(objective_sum)(target))))
batched = jax.vmap(
    lambda t: (
        alx.minimise(
            chain, solver, jnp.zeros(20), t, max_steps=4096, throw=False
        ).stats.infeasibility
    )
)
print("vmap ok:", jax.block_until_ready(batched(jnp.stack([target, 2 * target]))))

## Benchmark suites

Hock-Schittkowski and scaling problem sets with every first-order configuration plus the `baseline` and `newton` references, then the batched (`jax.vmap`) throughput benchmark.


In [ ]:
import subprocess
import sys

REPO = "/content/aulax"
RESULTS = "/content/results"

FIRST_ORDER = [
    "nonlinear_cg",
    "nonlinear_cg_gradient",
    "dog",
    "prodigy",
    "dadapt_adamw",
    "lion_1e-3",
    "lion_1e-4",
    "nonlinear_cg_tight_floor",
    "prodigy_tight_floor",
    "lbfgs_tight_floor",
    "lbfgs_tight_penalty_10",
]
REFERENCES = ["baseline", "newton"]
ALL_CONFIGS = FIRST_ORDER + REFERENCES


def run_benchmarks(problem_set, timeout):
    command = [
        sys.executable,
        "-m",
        "benchmarks.run",
        "--problems",
        problem_set,
        "--repeats",
        "3",
        "--workers",
        "2",
        "--timeout",
        str(timeout),
        "--output-dir",
        RESULTS,
    ]
    for config in ALL_CONFIGS:
        command += ["--config", config]
    subprocess.run(command, cwd=REPO, check=True)


run_benchmarks("hock_schittkowski", timeout=300)
run_benchmarks("scaling", timeout=600)

In [ ]:
BATCHED_CONFIGS = [
    "baseline",
    "newton",
    "nonlinear_cg",
    "dog",
    "prodigy",
    "dadapt_adamw",
]
for config in BATCHED_CONFIGS:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "benchmarks.batched",
            "--config",
            config,
            "--repeats",
            "3",
            "--output-dir",
            RESULTS,
        ],
        cwd=REPO,
        check=True,
    )

## Comparison tables


In [ ]:
import glob

for pattern in ["hock_schittkowski-*.json", "scaling-*.json"]:
    files = sorted(glob.glob(f"{RESULTS}/{pattern}"))
    print("\n=== " + pattern + " ===")
    subprocess.run(
        [sys.executable, "-m", "benchmarks.compare", *files], cwd=REPO, check=True
    )

In [ ]:
import json
from pathlib import Path

for path in sorted(Path(RESULTS).glob("batched-*.json")):
    content = json.loads(path.read_text())
    print(f"\n=== batched {content['configuration']} on {content['device']} ===")
    for record in content["records"]:
        print(
            f"{record['family']:24} batch {record['batch_size']:5d} "
            f"solved {record['solved']:5d}  {1000 * record['seconds']:10.2f} ms"
        )

## Download the results

Zips every result file in `/content/results` and downloads it.


In [ ]:
!cd /content && zip -q -r results.zip results && ls -la results.zip
from google.colab import files

files.download("/content/results.zip")